# Длительная эволюция гауссова импульса

Какие из пяти неустановившихся при $T=100$ состояний переходят к одному пику,
а какие сохраняют движение? Наблюдаем их до $T=200$ или $400$.

Начальное поле: $\psi(\theta,0)=A\exp[-\theta^2/(2\sigma^2)]$.
Параметры уравнения: $F=2$, $\Delta=3$, $\beta_2=-0.1$, $L=2\pi$;
сетка $N=512$, шаг $\Delta t=0.0005$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import Markdown, display
from lle_runs.project import get_experiment_dir
from lle_analysis.run_selection import load_runs_from_csv
from lle_solver.simulations.gaussian import GaussianConfig
from lle_solver.simulations.storage import load_evolution
from lle_analysis.criteria import DEFAULT_STATIONARY_TOLERANCE, is_stationary
from lle_analysis.fields import time_mask, relative_field_l2_error, spatial_contrast
from lle_analysis.peaks import peak_indices, peak_position
from lle_solver.grid import PeriodicGrid
from lle_viz.style import configure_plots, save_figure
from lle_viz.plots import plot_intensity, plot_profiles, plot_separation
from lle_viz.tables import table_markdown, save_table

In [ ]:
EXPERIMENT_DIR = get_experiment_dir(2)
RESULTS_DIR = EXPERIMENT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
configure_plots()

## Длительные наблюдения

Пять спорных точек рассчитаны до $T=200$ или $400$. Сначала проверим,
какие из них успевают стать стационарными.

[Запуски длительного наблюдения](followup_runs.csv): 5 расчётов.


In [ ]:
followup_runs = load_runs_from_csv(EXPERIMENT_DIR / "followup_runs.csv")

In [ ]:
followup_configs = {run.id: run.load_config(GaussianConfig) for run in followup_runs}
followup_evolutions = {run.id: load_evolution(run.path) for run in followup_runs}

### Измерения временного поведения

Для каждого запуска измеряем изменение поля, контраст и число пиков на последнем окне длительностью 20. Проверка стационарности относится только к этому окну. Далее отдельно исследуем движение пиков и изменение локальной формы.


In [ ]:
observation_duration = 20.
traces = {}

In [ ]:
followup = []
for run in followup_runs:
    config = followup_configs[run.id]
    evolution = followup_evolutions[run.id]
    tail = evolution[time_mask(evolution.times, evolution.times[-1] - observation_duration, evolution.times[-1])]
    traces[run.id] = dict(times=tail.times, field_error=relative_field_l2_error(tail, tail[-1]),
                          contrast=spatial_contrast(tail.intensity),
                          peak_counts=np.array([len(peak_indices(profile)) for profile in tail.intensity]))
    trace = traces[run.id]
    followup.append(dict(A=config.initial.amplitude, sigma=config.initial.width,
                         T=float(tail.times[-1]), window_start=float(tail.times[0]),
                         field_change=float(trace["field_error"].max()), stationary=is_stationary(tail),
                         contrast_max=float(trace["contrast"].max()),
                         peak_counts=sorted(set(trace["peak_counts"].tolist())), run_id=run.id))
followup.sort(key=lambda row: (row["A"], row["sigma"]))

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True, layout="constrained")
for row in followup:
    trace = traces[row["run_id"]]
    elapsed = trace["times"] - trace["times"][0]
    label = f'A={row["A"]:g}, sigma={row["sigma"]:.3g}, T={row["T"]:g}'
    axes[0].plot(elapsed, trace["field_error"], label=label)
    axes[1].plot(elapsed, trace["contrast"])
    axes[2].step(elapsed, trace["peak_counts"], where="mid")
axes[0].set(ylabel="Изменение поля")
axes[0].set_yscale("symlog", linthresh=1e-5)
axes[0].set_ylim(bottom=0)
axes[2].yaxis.set_major_locator(MaxNLocator(integer=True))
axes[2].set_ylim(min(line.get_ydata().min() for line in axes[2].lines) - .5,
                 max(line.get_ydata().max() for line in axes[2].lines) + .5)
axes[0].axhline(DEFAULT_STATIONARY_TOLERANCE, color="black", linestyle="--")
axes[0].legend(fontsize=8)
axes[1].set(ylabel="Контраст")
axes[2].set(xlabel="Время от начала окна", ylabel="Число пиков")
save_figure(RESULTS_DIR, "followup_time_metrics", figure)
plt.show()

### Движение пиков


In [ ]:
peak_coordinates = {}
for run in followup_runs:
    evolution = followup_evolutions[run.id]
    peak_coordinates[run.id] = []
    for field in evolution:
        intensity = field.intensity
        indices = peak_indices(intensity)
        peak_coordinates[run.id].append([peak_position(intensity, int(i)) for i in indices])

In [ ]:
positions = {}
for run in followup_runs:
    coordinates = peak_coordinates[run.id]
    positions[run.id] = np.full((len(coordinates), 2), np.nan)
    for index, peaks in enumerate(coordinates):
        if len(peaks) == 2:
            positions[run.id][index] = peaks

In [ ]:
separations = {}
for run in followup_runs:
    grid = PeriodicGrid.from_theta(followup_evolutions[run.id].theta)
    separations[run.id] = np.array([
        grid.distance(first, second) for first, second in positions[run.id]
    ])

### Форма в движущихся координатах

Сравниваем комплексное поле в окне $[-0.4,0.4]$ вокруг каждого пика
на последнем окне длительностью 20. Координаты уточнены параболой по трём узлам.


In [ ]:
offsets = np.linspace(-.4, .4, 129)
local_fields = {}
for run in followup_runs:
    evolution = followup_evolutions[run.id]
    indices = np.flatnonzero(time_mask(evolution.times, evolution.times[-1] - observation_duration, evolution.times[-1]))
    centers = positions[run.id][indices]
    if np.isnan(centers).any():
        continue
    local_fields[run.id] = [
        [evolution[index].relative_to(center).field_at(offsets) for center in row]
        for index, row in zip(indices, centers)
    ]

In [ ]:
shape_changes = {}
for run_id, snapshots in local_fields.items():
    final_profiles = snapshots[-1]
    shape_changes[run_id] = max(
        relative_field_l2_error(profile, final)
        for profiles in snapshots
        for profile, final in zip(profiles, final_profiles, strict=True)
    )

In [ ]:
motion = []
for row in followup:
    run_id = row["run_id"]
    evolution = followup_evolutions[run_id]
    tail = time_mask(evolution.times, evolution.times[-1] - observation_duration, evolution.times[-1])
    distance = separations[run_id][tail]
    has_pair = np.isfinite(distance).all()
    motion.append(dict(A=row["A"], sigma=row["sigma"], T=row["T"],
                       separation_start=float(distance[0]) if has_pair else None,
                       separation_end=float(distance[-1]) if has_pair else None,
                       separation_range=float(np.ptp(distance)) if has_pair else None,
                       shape_change=shape_changes.get(run_id), run_id=run_id))

### Эволюция пяти спорных случаев


In [ ]:
followup_intensities = {run.id: followup_evolutions[run.id].intensity for run in followup_runs}

#### Случай 1: Наблюдение ($A=2.4$, $\sigma=0.480137$, $T=200$)


In [ ]:
case_id = "2026-11-02_09-21-45_980295"
evolution = followup_evolutions[case_id]
intensity = followup_intensities[case_id]

profile_indices = [0, int(.8*len(evolution.times)), len(evolution.times)-1]
profiles = [intensity[i] for i in profile_indices]
profile_times = evolution.times[profile_indices]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel="$t$")
save_figure(RESULTS_DIR, f"evolution_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$t={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, f"profiles_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
plot_separation(ax, evolution.times, separations[case_id])
ax.set(xlabel="$t$", ylabel="Расстояние между пиками")
ax.grid(alpha=.2)
save_figure(RESULTS_DIR, f"separation_{case_id}", figure)
plt.show()

#### Случай 2: Наблюдение ($A=2.8$, $\sigma=0.480137$, $T=400$)


In [ ]:
case_id = "2026-11-02_09-28-14_701115"
evolution = followup_evolutions[case_id]
intensity = followup_intensities[case_id]

profile_indices = [0, int(.8*len(evolution.times)), len(evolution.times)-1]
profiles = [intensity[i] for i in profile_indices]
profile_times = evolution.times[profile_indices]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel="$t$")
save_figure(RESULTS_DIR, f"evolution_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$t={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, f"profiles_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
plot_separation(ax, evolution.times, separations[case_id])
ax.set(xlabel="$t$", ylabel="Расстояние между пиками")
ax.grid(alpha=.2)
save_figure(RESULTS_DIR, f"separation_{case_id}", figure)
plt.show()

#### Случай 3: Наблюдение ($A=2.8$, $\sigma=0.700000$, $T=400$)


In [ ]:
case_id = "2026-11-02_09-26-12_962949"
evolution = followup_evolutions[case_id]
intensity = followup_intensities[case_id]

profile_indices = [0, int(.8*len(evolution.times)), len(evolution.times)-1]
profiles = [intensity[i] for i in profile_indices]
profile_times = evolution.times[profile_indices]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel="$t$")
save_figure(RESULTS_DIR, f"evolution_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$t={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, f"profiles_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
plot_separation(ax, evolution.times, separations[case_id])
ax.set(xlabel="$t$", ylabel="Расстояние между пиками")
ax.grid(alpha=.2)
save_figure(RESULTS_DIR, f"separation_{case_id}", figure)
plt.show()

#### Случай 4: Наблюдение ($A=3.2$, $\sigma=0.329331$, $T=400$)


In [ ]:
case_id = "2026-11-02_09-26-12_965944"
evolution = followup_evolutions[case_id]
intensity = followup_intensities[case_id]

profile_indices = [0, int(.8*len(evolution.times)), len(evolution.times)-1]
profiles = [intensity[i] for i in profile_indices]
profile_times = evolution.times[profile_indices]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel="$t$")
save_figure(RESULTS_DIR, f"evolution_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$t={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, f"profiles_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
plot_separation(ax, evolution.times, separations[case_id])
ax.set(xlabel="$t$", ylabel="Расстояние между пиками")
ax.grid(alpha=.2)
save_figure(RESULTS_DIR, f"separation_{case_id}", figure)
plt.show()

#### Случай 5: Наблюдение ($A=3.2$, $\sigma=0.700000$, $T=400$)


In [ ]:
case_id = "2026-11-02_09-26-12_961949"
evolution = followup_evolutions[case_id]
intensity = followup_intensities[case_id]

profile_indices = [0, int(.8*len(evolution.times)), len(evolution.times)-1]
profiles = [intensity[i] for i in profile_indices]
profile_times = evolution.times[profile_indices]

In [ ]:
figure, ax = plt.subplots(layout="constrained")
image = plot_intensity(ax, intensity)
figure.colorbar(image, ax=ax, label=r"$|\psi|^2$")
ax.set(xlabel=r"$\theta$", ylabel="$t$")
save_figure(RESULTS_DIR, f"evolution_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
lines = plot_profiles(ax, profiles)
for line, time in zip(lines, profile_times):
    line.set_label(f"$t={time:g}$")
ax.set(xlabel=r"$\theta$", ylabel=r"$|\psi|^2$")
ax.legend()
save_figure(RESULTS_DIR, f"profiles_{case_id}", figure)
plt.show()

In [ ]:
figure, ax = plt.subplots(layout="constrained")
plot_separation(ax, evolution.times, separations[case_id])
ax.set(xlabel="$t$", ylabel="Расстояние между пиками")
ax.grid(alpha=.2)
save_figure(RESULTS_DIR, f"separation_{case_id}", figure)
plt.show()